In [52]:
import os
import json
import pandas as pd
import glob
from collections import Counter

cwd = os.getcwd()
data_dir = f"{cwd}/data"

#meta_df = pd.read_excel(data_dir+"/ASPECT_PolicieswIDs_3Mar26.xlsx")

In [27]:
all_jsons = glob.glob(f"{data_dir}/output_jsons/*")

We're gonna start with just one json but later we'll loop all the jsons in on this action

In [113]:
sents = []
for jsonf in all_jsons:
    with open(jsonf, encoding="utf-8") as f:
        mineru_pol = json.load(f)
    # each policy item is a list of page items
    # each page item is a dictionary containing the keys "page" and "content"
    # the "content" for each page is a list of content items
    # each content item is a dictionary containing the keys "type", "angle", and "content"
    # the "type" field options can be found here: https://github.com/opendatalab/mineru-vl-utils/blob/main/mineru_vl_utils/structs.py
    partial = False
    pol_id = mineru_pol[0]['policy_id'] 
    doc_id = jsonf.split("\\")[-1].split(".")[0]
    doc_name = mineru_pol[0]['document_name']
    chunk = 0
    for page_itm in mineru_pol[1:]:
        pg = str(page_itm['page'])
        for cont_itm in page_itm['content']:
            # reset partial to False after titles so that text before titles [which usually don't end with periods] 
            # don't merge with our first body text sentence
            if cont_itm['type']=="title":
                partial = False
            if cont_itm['type']=="text":
                text = cont_itm['content']
                if text:
                    if partial:
                        sents[-1]['text'] += " "+text
                        ls_pg = sents[-1]['pg'].split("-")
                        if ls_pg[-1] != pg:
                            sents[-1]['pg'] += f"-{pg}"
                        partial = False
                    else:
                        sents.append({"text": text, "pg": pg, "chunk_id":f'{doc_id}_{chunk}'})#, "pol_id": pol_id, "doc_id": doc_id})
                        chunk+=1
                    # if the text is incomplete (i.e. we will need to include the next content item)
                    if text[-1] != ".":
                        partial = True
                    else:
                        partial = False

In [114]:
print(len(sents))
sents

283810


[{'text': 'ALPENKONVENTION CONVENTION ALPINE ALPSKA KONVENCIJA CONVENZIONDELLALPI',
  'pg': '1',
  'chunk_id': '1009_0_0'},
 {'text': 'Herzog-Friedrich-StraBe 15 6020 Innsbruck Austria',
  'pg': '2',
  'chunk_id': '1009_0_1'},
 {'text': 'Viale Druso / Drususallee 1 39100 Bolzano / Bozen Italy The Climate Action Plan 2.0 was developed by the Alpine Climate Board of the Alpine Convention, on the basis of a draft prepared by Helen Lückge (Climonomics) and with inputs from the Contracting Parties, Observers, Thematic Working Bodies, the Permanent Secretariat of the Alpine Convention and other experts. The work of the Alpine Climate Board of the Alpine Convention was funded by Austria, Germany and Switzerland.',
  'pg': '2',
  'chunk_id': '1009_0_2'},
 {'text': 'www.alpineclimate2050.org info@alpineclimate2050.org Translations: ALPS-LaRete Graphic design: Mauro Sutter Design Printing: Sterndruck, Fugen, Austria Photographs: Moritz Kaser (cover photo), Hannes Schlosser (transport), Alessandr

Let's loop

In [115]:
with open(f"{data_dir}/peat_doc_sents.jsonl", "w", encoding="utf-8") as f:
    for sample in sents:
        json.dump(sample, f)
        f.write("\n")

In [116]:
with open(f"{data_dir}/peat_doc_sents.jsonl", "r", encoding="utf-8") as f:
    pged_data = [json.loads(line) for line in f]

In [117]:
pged_data = sents

metric_entities = []
peat_entities = []

for entity in pged_data:
    text = entity['text'].lower()
    if "peat" in text or "bog" in text:
        peat_entities.append(entity['chunk_id'])

for entity in pged_data:
    text = entity['text'].lower()
    if "ha" in text or "km" in text or "area" in text:
        metric_entities.append(entity['chunk_id'])
    if "emission" in text or "ghg" in text:
        metric_entities.append(entity['chunk_id'])
    if "site" in text:
        metric_entities.append(entity['chunk_id'])
    if "budget" in text or "€" in text:
        metric_entities.append(entity['chunk_id'])
    if "resource" in text or "report" in text:
        metric_entities.append(entity['chunk_id'])
    if "prohibit" in text or "law" in text or "scheme" in text:
        metric_entities.append(entity['chunk_id'])
    if "train" in text or "teach" in text or "scheme" in text:
        metric_entities.append(entity['chunk_id'])

print(len(metric_entities), len(peat_entities))
metric_entities = set(metric_entities)
peat_entities = set(peat_entities)
print(len(metric_entities), len(peat_entities))

190200 3587
155946 3587


In [118]:
metric_docs = []
peat_docs = []
for entry in metric_entities:
    metric_docs.append(int(entry.split("_")[0]))
for entry in peat_entities:
    peat_docs.append(int(entry.split("_")[0]))
md = list(dict(Counter(metric_docs)).items())
pd = list(dict(Counter(peat_docs)).items())

In [119]:
metric_docs = set(metric_docs)
peat_docs = set(peat_docs)
print(len(metric_docs), len(peat_docs))
common = []
for doc in metric_docs:
    if doc in peat_docs:
        common.append(doc)
print(len(common))

285 190
190


In [120]:
md = sorted(md, key=lambda tup: tup[1], reverse=True)
pd = sorted(pd, key=lambda tup: tup[1], reverse=True)
m = [entry[0] for entry in md]
p = [entry[0] for entry in pd]

In [121]:
n = 20
# 10: 319
# 15: 304, 319
# 20: 304, 293, 285, 319
list(set(m[:n]).intersection(p[:n]))

[304, 293, 285, 319]

In [126]:
filtered_entities = []
for entry in pged_data:
    if int(entry['chunk_id'].split("_")[0]) in [319]:#[304, 293, 285, 319]:
        entry["label"]=[]
        filtered_entities.append(entry)
len(filtered_entities)

4446

In [127]:
filtered_entities

[{'text': 'Type of modification 23 General information on the request for amendment 23 Type of amendment 23 Detailed information on the specific elements of each modification 24 Amendment to AECM General and Training to Implement Agri-Environment-Climate Measure.24 Reasonsthat justify the change 24 Expected effects of the change 24 The impact of the change on targets and indicators 24 The impact of the change on the financing plan. 24 Amendment to GAEC 2 .24 Reasonsthat justify the change 24 Expected effects of the change 25 The impact of the change on targets and indicators 25 The impact of the change on the financing plan. 25 Changes to Annex 7.3 EAFRD.. 26 Reasonsthat justify the change 26 Expected effects of the change 26 The impact of the change on targets and indicators 26 The impact of the change on the financing plan. 26 Changes to Annex 7.3 EAGF 26 Reasonsthat justify the change 26 Expected effects of the change 26 The impact of the change on targets and indicators 26 The impa

In [128]:
with open(f"{data_dir}/ie_cap.jsonl", "w", encoding="utf-8") as f:
    for sample in filtered_entities:
        json.dump(sample, f)
        f.write("\n")

In [3]:
import os
import json
cwd = os.getcwd()
with open(f"{cwd}/data/peat_doc_sents.jsonl", "r", encoding="utf-8") as f:
    s_json_list = list(f)

s_set= []
for json_str in s_json_list:
    result = json.loads(json_str)
    s_set.append(result)

In [4]:
len(s_set)

283810